# B2 — 120 pair test tezgâhı (v4, 11 Eki: #95 dersleri)
Ressam (45K parametre, ~45 KB) + pair başına notlar (1/64, 1/32, 1/16 ızgaraları, entropi kodlu).

**Yöntem:** yöntemleri **120 pair**'de yan yana dene, kazananı **600 pair**'de bir kez doğrula. Epoch sayısı kısaltılmaz (bit aşaması 1500 + net_ft 600).

**v4'te yeni** (kaynak: #95 `hnerv_muon` eğitim kodu → proje notu `pr95-egitim-karsilastirmasi-2026-10-11.md`):
- **EMA:** ağırlıkların ve notların hareketli ortalaması (`ema_ep` = ufuk, epoch cinsinden). Değerlendirme ve kayıt EMA kopyasıyla yapılır.
- **Not LR çarpanı** (`not_lr`): bit aşamasında notların öğrenme hızı. #95 bu aşamada latent LR'yi çok düşük tutuyor.
- **En iyi checkpoint:** `net_ft` her 50 epoch'ta ölçer ve en iyisini tutar. Seçim **8 bit ressamlı** dist'e göre (`dist8`), çünkü gerçek arşivdeki ressam 8 bit.
- **Ressam QAT** (`qat=True`, isteğe bağlı): `net_ft` ileri geçişi 8 bit ressamla yapar.
- **Mimari koşu başına** (`levels`, `widths`); FP32 başlangıcı sıfırdan eğitilebilir (`fp32`, `fp32_ep`). Ressam KB'ı parametre sayısından hesaplanır.
- **Batch GPU belleğine göre:** L4 (22 GB) → net_ft batch 20; A100 40 GB → 40. v3'te L4'te batch 40 OOM veriyordu.

**Kurallar:** karar toplam skora göre (dist8 ile) · her 10 KB ≈ 0.0067 · ara satırlar yanıltıcı, `SON` satırı esas · her koşu yeni etiket.

**Çalıştırma sırası:** Bölüm 1–7 → Bölüm 8 döngü hücresi → 8.1 → 8.2 → (8.3).

### Durum (11 Eki)
| Koşu | Pair | Dist | Notlar | Toplam | Oran |
|---|---|---|---|---|---|
| p600_uzun (FP32, bit yok) | 600 | **0.0835** | — | — | — |
| p200u (sert, λ30) | 200 | 0.2483 | 94.6 KB | **0.341** | 2.8x |
| p600c3 (C3, λ30) | 600 | 0.3033 | 78.3 KB | 0.385 | 3.6x |
| p600sifir (C3, notlar sıfırdan) | 600 | 0.3342 | 65.1 KB | 0.408 | 4.0x |
| t120c3 (C3, λ30) — yalnız bit aşaması, net_ft OOM | 120 | 0.2803 | 97 KB | (0.375) | — |

v3 kuyruğunda `t120u` ve `t120c3`'ün bit aşamaları bitti ama `net_ft` L4'te OOM verdi → 8.1 bunları tamamlar (yalnız net_ft, bit dosyaları var).

**#95 kıyası:** dist ≈ 0.073 (seg 0.056 + pose 0.017), arşiv ~178 KB → 0.199. FP32'de biz de 0.0835'teyiz; açık tamamen nicemlemede.

## 1. Kurulum

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import torch
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| cihaz:", device)
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name, f"| bellek: {props.total_memory / 2**30:.1f} GB")

ROOT = Path("/content/drive/MyDrive/deneme")
VIDEO_PATH = Path("/content/drive/MyDrive/0.mkv")
MODEL_DIR = ROOT / "models"
TARGET_CACHE = ROOT / "checkpoints" / "hnerv384_targets_evalrgb_v1.pt"
OUT_DIR = ROOT / "b2"                      # bu notebook'un bütün çıktıları buraya
OUT_DIR.mkdir(exist_ok=True)

for p in (VIDEO_PATH, MODEL_DIR / "posenet.safetensors", MODEL_DIR / "segnet.safetensors", TARGET_CACHE):
    print("OK " if p.exists() else "YOK", p)

Mounted at /content/drive
PyTorch: 2.11.0+cu130 | cihaz: cuda
GPU: NVIDIA A100-SXM4-40GB | bellek: 39.5 GB
OK  /content/drive/MyDrive/0.mkv
OK  /content/drive/MyDrive/deneme/models/posenet.safetensors
OK  /content/drive/MyDrive/deneme/models/segnet.safetensors
OK  /content/drive/MyDrive/deneme/checkpoints/hnerv384_targets_evalrgb_v1.pt


In [2]:
import gc, sys, traceback, types, warnings, torch, torch.nn as nn

KORU = {"distortion_net", "segnet", "posenet", "seg_targets", "pose_targets", "PILOT", "device"}  # silinmeyecekler

def _gpu_tutuyor(o, d=0):
    if torch.is_tensor(o): return o.is_cuda
    if isinstance(o, nn.Module): return any(t.is_cuda for t in [*o.parameters(), *o.buffers()])
    if isinstance(o, torch.optim.Optimizer):
        return any(torch.is_tensor(v) and v.is_cuda for s in o.state.values() for v in s.values()) \
            or any(p.is_cuda for g in o.param_groups for p in g["params"])
    if d < 2 and isinstance(o, dict): return any(_gpu_tutuyor(v, d + 1) for v in list(o.values())[:500])
    if d < 2 and isinstance(o, (list, tuple)): return any(_gpu_tutuyor(v, d + 1) for v in list(o)[:500])
    return False

def _kareleri_bosalt(exc, gorulen=None):
    gorulen = gorulen or set()
    while exc is not None and id(exc) not in gorulen:            # hata zinciri: cause / context
        gorulen.add(id(exc))
        if exc.__traceback__ is not None:
            traceback.clear_frames(exc.__traceback__)            # karelerdeki yerel değişkenleri sil
        _kareleri_bosalt(exc.__cause__, gorulen)
        exc = exc.__context__

def gpu_temizle(*isimler, koru=()):
    """gpu_temizle() → GPU tutan tüm globalleri sil (KORU hariç). İsim verirsen onları da siler."""
    try: torch.cuda.synchronize()                                # yarıda kalan GPU işini bitir
    except Exception: pass
    # 1) donmuş traceback kareleri (Durdur / OOM / hata)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")                          # torch'un eski isimleri uyarı basmasın
        for o in gc.get_objects():                               # kim saklarsa saklasın: tüm traceback'ler
            try:
                if isinstance(o, types.TracebackType):
                    traceback.clear_frames(o)                    # karelerin yerellerini boşalt
            except ReferenceError:                               # ölü weakref proxy: dokununca patlıyor, atla
                continue
    ip = get_ipython()
    hatalar = [getattr(sys, "last_exc", None), getattr(sys, "last_value", None)]
    if getattr(ip, "last_execution_result", None) is not None:
        hatalar.append(ip.last_execution_result.error_in_exec)
        ip.last_execution_result.error_in_exec = None
    for e in hatalar:
        if isinstance(e, BaseException): _kareleri_bosalt(e)
    if getattr(sys, "last_traceback", None) is not None:
        traceback.clear_frames(sys.last_traceback)
    sys.last_exc = sys.last_type = sys.last_value = sys.last_traceback = None
    # 2) globaller: verilen isimler + GPU tutan her şey (korunanlar hariç)
    g = ip.user_ns
    sil = set(isimler) | {n for n, v in list(g.items())
                          if not n.startswith("_") and n not in KORU | set(koru) and _gpu_tutuyor(v)}
    for n in sorted(sil):
        g.pop(n, None)
    print("silindi:", ", ".join(sorted(sil)) or "—")
    # 3) Out[] geçmişi ve _ / __ / ___
    ip.run_line_magic("reset", "-f out")
    # 4) topla ve rafı geri ver
    gc.collect()
    torch.cuda.empty_cache(); torch.cuda.ipc_collect()


def gpu_durum(etiket=""):
    a, r = torch.cuda.memory_allocated() / 2**30, torch.cuda.memory_reserved() / 2**30
    t = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"[GPU {etiket}] kullanılan {a:.1f} GB | ayrılmış {r:.1f} GB | toplam {t:.1f} GB")

## 2. Değerlendiriciler ve hedefler (SegNet / PoseNet donmuş)

In [3]:
!uv pip -q install segmentation-models-pytorch timm einops safetensors av

import sys
sys.path.insert(0, str(ROOT))                      # research/ klasörü Drive'da, deneme/ altında
from research.modules import DistortionNet
from research.oracle_pair import seg_preprocess_grad, pose_preprocess_grad

distortion_net = DistortionNet().eval().to(device)
distortion_net.load_state_dicts(MODEL_DIR / "posenet.safetensors", MODEL_DIR / "segnet.safetensors", device)
distortion_net.requires_grad_(False)               # ağırlıkları asla değişmeyecek

n_seg = sum(p.numel() for p in distortion_net.segnet.parameters())
n_pose = sum(p.numel() for p in distortion_net.posenet.parameters())
print(f"SegNet {n_seg / 1e6:.1f}M | PoseNet {n_pose / 1e6:.1f}M parametre (donmuş)")

SegNet 9.5M | PoseNet 13.9M parametre (donmuş)


In [4]:
targets = torch.load(TARGET_CACHE, map_location="cpu", weights_only=True)
seg_targets = targets["seg_class"].to(device, dtype=torch.uint8)    # [600, 384, 512]: piksel başına sınıf
pose_targets = targets["pose"].to(device, dtype=torch.float32)      # [600, 6]: pair başına kamera hareketi
NUM_PAIRS = seg_targets.shape[0]
print("Seg:", tuple(seg_targets.shape), "| Pose:", tuple(pose_targets.shape), "| pair:", NUM_PAIRS)

counts = torch.bincount(seg_targets.flatten().long(), minlength=5)
for c, n in enumerate(counts.tolist()):
    print(f"  class {c}: %{100 * n / counts.sum().item():6.2f}")
print("Pose std (boyut başına):", [round(v, 4) for v in pose_targets.std(0).tolist()])

Seg: (600, 384, 512) | Pose: (600, 6) | pair: 600
  class 0: % 23.23
  class 1: %  0.59
  class 2: % 49.52
  class 3: %  1.24
  class 4: % 25.43
Pose std (boyut başına): [1.2574, 0.0358, 0.0299, 0.0096, 0.0074, 0.0286]


## 3. Skor (resmi formül, FP32, TF32 kapalı)

In [5]:

import torch.nn.functional as F

def inflate(native):
    """[B, 2, 3, 384, 512] → [B, 2, 3, 874, 1164]. İleri geçişte yuvarlar, geri geçişte gradyanı geçirir."""
    b = native.shape[0]
    up = F.interpolate(native.flatten(0, 1), size=(874, 1164), mode="bicubic", align_corners=False)
    up = up.clamp(0, 255)
    up = up + (up.round() - up).detach()          # STE: ileri = round, geri = özdeşlik
    return up.view(b, 2, 3, 874, 1164)


@torch.no_grad()
def score(render, ids_all, bs=16):
    """render(ids) → [B, 2, 3, 384, 512] (0–255). Resmi ölçümle aynı ayar: FP32, TF32 kapalı."""
    old = (torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32)
    torch.backends.cuda.matmul.allow_tf32 = torch.backends.cudnn.allow_tf32 = False
    wrong, pix, pose_sq = 0, 0, 0.0
    for s in range(0, len(ids_all), bs):
        ids = ids_all[s:s + bs]
        full = inflate(render(ids))
        pred = distortion_net.segnet(seg_preprocess_grad(full)).argmax(1)
        pose = distortion_net.posenet(pose_preprocess_grad(full))["pose"][:, :6].float()
        wrong += int((pred != seg_targets[ids].long()).sum())
        pix += pred.numel()
        pose_sq += float((pose - pose_targets[ids]).square().mean(1).sum())
    torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32 = old
    seg, pose_t = 100 * wrong / pix, (10 * pose_sq / len(ids_all)) ** 0.5
    return {"dist": seg + pose_t, "seg": seg, "pose": pose_t}


PILOT = torch.arange(120, device=device)         # test tezgâhı: ilk 120 pair (tam video 600)

## 4. Model ve loss

In [6]:
import math
import torch.nn as nn
GRID_LEVELS = {64:2, 32:2, 16:2}
WIDTHS = [16, 16, 16, 16, 12, 12, 8]

class GridDecoder(nn.Module):
  def __init__(self, n_pairs, levels = GRID_LEVELS, widths = WIDTHS):
    super().__init__()
    self.grids = nn.ParameterDict({str(d): nn.Parameter(torch.zeros(n_pairs, c, 384 // d, 512 // d))
                                  for d, c in levels.items()})

    self.inject = nn.ModuleDict({str(d): nn.Conv2d(c, widths[int(math.log2(64 // d))], 1)
                                     for d, c in levels.items()})
    self.ups = nn.ModuleList(nn.Conv2d(widths[i], 4 * widths[i + 1], 3, padding=1) for i in range(6))
    self.skips = nn.ModuleList(nn.Conv2d(widths[i], widths[i + 1], 1) for i in range(6))
    self.head = nn.Conv2d(widths[-1], 6, 3, padding=1)
  def forward(self, ids, grids = None):
    g = grids if grids is not None else {k: v[ids] for k, v in self.grids.items()}
    x = self.inject["64"](g["64"])
    for i in range(6):
      up = F.pixel_shuffle(self.ups[i](x), 2)
      skip = F.interpolate(self.skips[i](x), scale_factor = 2, mode = "bilinear", align_corners = False)
      x = F.gelu(up + skip)
      d = str(64//2 ** (i+1))
      if d in self.inject:
        x = x + self.inject[d](g[d])
    return (torch.sigmoid(self.head(x)) * 255).view(len(ids), 2, 3, 384, 512)

In [7]:

def l7_loss(logits, truth, tau = 0.3,thr =1.0, mult = 4.0):
  correct = logits.gather(1, truth.unsqueeze(1))
  others = logits.clone()
  others.scatter_(1, truth.unsqueeze(1), torch.finfo(others.dtype).min)
  margin = correct - others.amax(1, keepdim = True)
  per_pixel = tau * F.softplus(-margin / tau)

  with torch.no_grad():
    w = 1.0 + mult * (margin < thr).float()
    w = w/w.mean()

  return (per_pixel * w).mean()

def task_loss(native, ids):
  full = inflate(native)
  logits = distortion_net.segnet(seg_preprocess_grad(full))
  pose = distortion_net.posenet(pose_preprocess_grad(full))["pose"][:, :6].float()
  seg_l = 100 * l7_loss(logits, seg_targets[ids].long())
  pose_l = torch.sqrt(10 * F.mse_loss(pose, pose_targets[ids]) + 1e-6)
  return seg_l + pose_l, seg_l.item(), pose_l.item()

## 5. Bit / boyut araçları ve yuvarlama (sert, yumuşak, C3)

In [8]:
BIT2SCORE = 25 / 8 / 37545489                     # 1 bitin skordaki karşılığı
NET_KB = 45                                       # ressam, 8 bit (entropi kodu yalnızca 44.1 KB'a indiriyor)
rate_of_kb = lambda kb: 25 * kb * 1e3 / 37545489  # KB → skor


@torch.no_grad()
def quant_net_(m, bits=8):
    """Ressamın conv ağırlıklarını yerinde, kanal başına simetrik `bits` bite yuvarlar."""
    qmax = 2 ** (bits - 1) - 1
    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            s = mod.weight.abs().amax(dim=(1, 2, 3), keepdim=True).clamp_min(1e-12) / qmax
            mod.weight.copy_(torch.round(mod.weight / s).clamp(-qmax, qmax) * s)


def entropy_kb(q):
    """Tamsayı dizisinin ampirik entropisi (değerler bağımsız) → KB."""
    _, counts = torch.unique(q, return_counts=True)
    p = counts.double() / q.numel()
    return float(-(counts * p.log2()).sum()) / 8 / 1e3


def laplace_bits(y, b):
    """y değerinin (±0.5 kutusu) Laplace(0, b) altında bit maliyeti.
    DÜZELTME (6 Eki): p_in'deki işaret '+' olmalı; eski sürümde '-' idi → sıfıra yakın değerler 0 bit sayılıyordu."""
    a = y.abs()
    a_in, a_out = a.clamp(max=0.5), a.clamp(min=0.5)
    p_in = 1 - 0.5 * (torch.exp(-(0.5 - a_in) / b) + torch.exp(-(0.5 + a_in) / b))
    p_out = 0.5 * (torch.exp(-(a_out - 0.5) / b) - torch.exp(-(a_out + 0.5) / b))
    return -torch.log2(torch.where(a < 0.5, p_in, p_out).clamp_min(1e-9))


def grids_for(m, ids, train):
    """Eğitimde: ileri geçiş yuvarlanmış değeri görür (STE) + bit tahmini için gürültülü kopya.
       Değerlendirmede: düz yuvarlama."""
    fwd, noisy = {}, {}
    for k, g in m.grids.items():
        x = g[ids]
        fwd[k] = x + (torch.round(x) - x).detach() if train else torch.round(x)
        noisy[k] = x + torch.rand_like(x) - 0.5
    return fwd, noisy


def soft_round(y, alpha):
    """alpha küçük → neredeyse y (düz çizgi); alpha büyük → round(y) (merdiven)."""
    m = torch.floor(y) + 0.5                       # bulunduğu kutunun ortası (ör. 0.3 → 0.5)
    r = y - m                                      # kutu içindeki konum: -0.5 .. +0.5
    return m + torch.tanh(alpha * r) / (2 * math.tanh(alpha / 2))


def grids_for_soft(m, ids, alpha):
    """Eğitimde yumuşak yuvarlama: ressam s(s(y) + gürültü)'yü görür, bit tahmini s(y) + gürültü'den."""
    fwd, noisy = {}, {}
    for k, g in m.grids.items():
        s = soft_round(g[ids], alpha)
        noisy[k] = s + torch.rand_like(s) - 0.5
        fwd[k] = soft_round(noisy[k], alpha)
    return fwd, noisy


def kumaraswamy_noise(shape, a, device):
    """[-0.5, 0.5] aralığında gürültü. a = 1 → düz (bizim eski ±0.5); a > 1 → ortaya yığılmış."""
    b = (2 ** a * (a - 1) + 1) / a                       # tepe tam 0.5'te olsun diye b seçilir
    u = torch.rand(shape, device=device).clamp(1e-6, 1 - 1e-6)
    x = (1 - (1 - u) ** (1 / b)) ** (1 / a)              # ters CDF ile örnekleme → [0, 1]
    return x - 0.5                                       # → [-0.5, 0.5]


def grids_for_c3(m, ids, alpha, a):
    """C3 1. aşama: bit tahmini s(y) + gürültü'den, ressam s(s(y) + gürültü)'yü görür."""
    fwd, noisy = {}, {}
    for k, g in m.grids.items():
        s = soft_round(g[ids], alpha)
        noisy[k] = s + kumaraswamy_noise(s.shape, a, s.device)
        fwd[k] = soft_round(noisy[k], alpha)
    return fwd, noisy

## 6. Bağlamlı entropi modeli (C3 A.1.4 tarzı)
Her not için komşularından (5×5 nedensel maske) ve bir kaba seviyeden μ, σ tahmini → Laplace bit. Sıra 1/64 → 1/32 → 1/16. Şimdilik **sonradan** uygulanıyor (birlikte eğitim notları kötüleştirdi).

In [9]:
class MaskedConv2d(nn.Conv2d):
  def __init__(self, c_in, c_out, k = 5):
    super().__init__(c_in, c_out, k, padding = k//2)
    mask = torch.ones(k, k)
    mask[k // 2, k // 2:] = 0
    mask[k // 2 + 1:,:] = 0
    self.register_buffer("mask", mask.view(1, 1, k, k))
  def forward(self, x):
    return F.conv2d(x, self.weight * self.mask, self.bias, padding = self.padding)

In [10]:
class LevelEntropy(nn.Module):
  def __init__(self, c = 2, h = 16, has_coarse = True):
    super().__init__()
    self.same = MaskedConv2d(c, h, 5)
    self.coarse = nn.Conv2d(c, h, 1) if has_coarse else None
    self.mix = nn.Sequential(
        nn.GELU(), nn.Conv2d(h, h, 1),
        nn.GELU(), nn.Conv2d(h, 2 * c, 1),
    )
    nn.init.zeros_(self.mix[-1].weight)
    nn.init.zeros_(self.mix[-1].bias)
  def forward(self, q, q_coarse):
    x = self.same(q)
    if self.coarse is not None:
      up = F.interpolate(q_coarse, size=q.shape[-2:], mode="nearest")
      x = x + self.coarse(up)
    mu, log_s = self.mix(x).chunk(2, dim = 1)
    return mu, log_s.exp().clamp(0.1)


class ContextEntropy(nn.Module):
  ORDER = ["64", "32", "16"]
  def __init__(self, levels = GRID_LEVELS):
    super().__init__()
    self.nets = nn.ModuleDict({k: LevelEntropy(levels[int(k)], has_coarse=(i > 0))
                            for i, k in enumerate(self.ORDER)})
  def forward(self, q):
    out = {}
    for i, k in enumerate(self.ORDER):
      onceki = q[self.ORDER[i-1]] if i>0 else None
      out[k] = self.nets[k](q[k], onceki)
    return out

In [11]:
ent_kb = lambda e: sum(p.numel() for p in e.parameters()) / 1e3        # entropi ağı ≈ parametre başına 1 bayt

def context_bits(ent, ctx, vals):
    """ctx: bağlam olarak kullanılan yuvarlanmış notlar. vals: biti ölçülen değerler
    (eğitimde gürültülü, ölçümde tamsayı). Döner: {seviye: toplam bit}."""
    tahmin = ent(ctx)
    return {k: laplace_bits(vals[k] - mu, s).sum() for k, (mu, s) in tahmin.items()}


@torch.no_grad()
def context_kb(ent, grids, ids_all, bs=20):
    """Tamsayı notların bağlamlı model altındaki kod uzunluğu, 600 pair'e ölçekli, seviye seviye KB."""
    toplam = {k: 0.0 for k in ContextEntropy.ORDER}
    for s in range(0, len(ids_all), bs):
        ids = ids_all[s:s + bs]
        q = {k: torch.round(grids[k][ids]) for k in ContextEntropy.ORDER}
        for k, b in context_bits(ent, q, q).items():
            toplam[k] += b.item()
    olcek = NUM_PAIRS / len(ids_all) / 8 / 1e3                       # bit → KB, 600 pair'e ölçekle
    return {k: v * olcek for k, v in toplam.items()}

## 7. Eğitim fonksiyonları

In [12]:
import time
from tqdm.auto import tqdm


def run_pilot(levels, widths, epochs = 300, tag = "", batch = 20, lr_net=1e-3, lr_grid=1e-2, eval_every=100):
  torch.manual_seed(0)                                       # her deney aynı başlangıçla → adil kıyas
  m = GridDecoder(NUM_PAIRS, levels, widths).to(device)
  n_net = sum(p.numel() for n, p in m.named_parameters() if not n.startswith("grids"))
  n_grid = sum(g[0].numel() for g in m.grids.values())
  print(f"[{tag}] ağ {n_net:,} parametre | pair başına {n_grid:,} ızgara değeri")
  net_p = [p for n, p in m.named_parameters() if not n.startswith("grids")]
  opt = torch.optim.Adam([{"params": net_p, "lr": lr_net},
                          {"params": list(m.grids.parameters()), "lr": lr_grid}])
  total = epochs * math.ceil(len(PILOT) / batch)
  sched = torch.optim.lr_scheduler.LambdaLR(
      opt, lambda s: min(1.0, (s + 1) / 100) * 0.5 * (1 + math.cos(math.pi * min(s / total, 1.0))))
  t0 = time.time()

  for ep in tqdm(range(1, epochs + 1)):
    m.train()
    perm = PILOT[torch.randperm(len(PILOT), device=device)]
    for s in range(0, len(perm), batch):
            ids = perm[s:s + batch]
            loss, _, _ = task_loss(m(ids), ids)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
            sched.step()
    if ep % eval_every == 0:
            r = score(lambda i: m(i), PILOT)
            print(f"[{tag}] E{ep:04d} | dist {r['dist']:.4f} (seg {r['seg']:.4f}, pose {r['pose']:.4f}) | "
                  f"{(time.time() - t0) / 60:.1f} dk", flush=True)
  torch.save(m.state_dict(), OUT_DIR / f"pilot_{tag}.pt")
  del m, opt
  torch.cuda.empty_cache()
  return r


def fp32_devam(init_path, tag, epochs, batch=20, lr_net=5e-4, lr_grid=5e-3, warm=20, eval_every=250):
    """FP32 checkpoint'ten devam (bit yok): yarım LR'de yeniden ısınma + kosinüs. Her ölçümde kaydeder."""
    torch.manual_seed(0)
    m = GridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device)
    m.load_state_dict(torch.load(init_path, map_location=device))
    net_p = [p for n, p in m.named_parameters() if not n.startswith("grids")]
    opt = torch.optim.Adam([{"params": net_p, "lr": lr_net},
                            {"params": list(m.grids.parameters()), "lr": lr_grid}])
    adim = math.ceil(len(PILOT) / batch)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / (warm * adim))
                                              * 0.5 * (1 + math.cos(math.pi * min(1, s / (epochs * adim)))))
    r = score(m, PILOT)
    print(f"[{tag}] başlangıç | dist {r['dist']:.4f} (seg {r['seg']:.4f}, pose {r['pose']:.4f})")
    t0 = time.time()
    for ep in tqdm(range(epochs)):
        perm = PILOT[torch.randperm(len(PILOT), device=device)]
        for i in range(0, len(perm), batch):
            ids = perm[i:i + batch]
            loss, _, _ = task_loss(m(ids), ids)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step(); sched.step()
        if (ep + 1) % eval_every == 0:
            r = score(m, PILOT)
            print(f"[{tag}] E{ep + 1:04d} | dist {r['dist']:.4f} (seg {r['seg']:.4f}, pose {r['pose']:.4f}) "
                  f"| {(time.time() - t0) / 60:.1f} dk", flush=True)
            torch.save(m.state_dict(), OUT_DIR / f"pilot_{tag}.pt")
    return r

### 7.0 v4 yardımcıları: EMA, 8 bit QAT, sıfırdan FP32

In [13]:
import copy


class EMA:
    """Ağırlıkların hareketli ortalaması (#95: 0.999).
    Gölge kopya her adımda canlı modele biraz yaklaşır → yuvarlama gürültüsüyle zıplayan değerleri yumuşatır.
    ufuk_adim: gölgenin kabaca kaç adımı 'hatırladığı' (decay = 1 - 1/ufuk)."""
    def __init__(self, model, ufuk_adim):
        self.decay = 1 - 1 / max(1, ufuk_adim)
        self.golge = {k: v.detach().clone() for k, v in model.state_dict().items()}
        self.n = 0

    @torch.no_grad()
    def guncelle(self, model):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))          # başta hızlı ısınma
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.golge[k].lerp_(v, 1 - d)                       # golge = d·golge + (1-d)·v
            else:
                self.golge[k].copy_(v)

    def kopya(self, sablon):
        """Gölge ağırlıklarla yeni bir model (değerlendirme / kayıt için)."""
        m = copy.deepcopy(sablon)
        m.load_state_dict(self.golge)
        return m


def qat_ac(m, bits=8):
    """Ressamın conv ağırlıklarını geçici olarak 8 bit (kanal başına) yuvarlanmış halleriyle değiştirir.
    İleri ve geri geçiş yuvarlanmış ağırlığı görür; adım asıl (FP32) ağırlığa uygulanır (STE, #95 gibi).
    Dönen sözlüğü backward'dan SONRA qat_kapat'a ver."""
    qmax = 2 ** (bits - 1) - 1
    asil = {}
    for ad, mod in m.named_modules():
        if isinstance(mod, nn.Conv2d):
            w = mod.weight.data
            s = w.abs().amax(dim=(1, 2, 3), keepdim=True).clamp_min(1e-12) / qmax
            asil[ad] = w
            mod.weight.data = torch.round(w / s).clamp(-qmax, qmax) * s
    return asil


def qat_kapat(m, asil):
    for ad, mod in m.named_modules():
        if ad in asil:
            mod.weight.data = asil[ad]


def ressam_kb(m):
    """8 bit ressam ≈ parametre başına 1 bayt (entropi kodu biraz daha küçültür; temkinli tahmin)."""
    return sum(p.numel() for n, p in m.named_parameters() if not n.startswith("grids")) / 1e3


def fp32_sifirdan(tag, epochs=2400, batch=20, lr_net=1e-3, lr_grid=1e-2, warm=20, ema_ep=25, eval_every=400):
    """Rastgele başlangıçtan FP32 eğitimi (bit yok), PILOT pair'lerinde, GRID_LEVELS / WIDTHS mimarisiyle.
    Kayıt: pilot_{tag}.pt (EMA ağırlıkları). Test B'de iki mimariyi aynı başlangıç koşullarında kıyaslamak için."""
    torch.manual_seed(0)
    m = GridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device)
    net_p = [p for n, p in m.named_parameters() if not n.startswith("grids")]
    opt = torch.optim.Adam([{"params": net_p, "lr": lr_net},
                            {"params": list(m.grids.parameters()), "lr": lr_grid}])
    adim = math.ceil(len(PILOT) / batch)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / (warm * adim))
                                              * 0.5 * (1 + math.cos(math.pi * min(1, s / (epochs * adim)))))
    ema = EMA(m, ema_ep * adim) if ema_ep else None
    print(f"[{tag}] FP32 sıfırdan | {len(PILOT)} pair | ressam {ressam_kb(m) * 1e3:,.0f} parametre | {epochs} epoch")
    t0 = time.time()
    for ep in tqdm(range(epochs)):
        perm = PILOT[torch.randperm(len(PILOT), device=device)]
        for i in range(0, len(perm), batch):
            ids = perm[i:i + batch]
            loss, _, _ = task_loss(m(ids), ids)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step(); sched.step()
            if ema: ema.guncelle(m)
        if (ep + 1) % eval_every == 0 or ep + 1 == epochs:
            em = ema.kopya(m) if ema else m
            r = score(em, PILOT)
            print(f"[{tag}] E{ep + 1:04d} | dist {r['dist']:.4f} (seg {r['seg']:.4f}, pose {r['pose']:.4f}) "
                  f"| {(time.time() - t0) / 60:.1f} dk", flush=True)
            torch.save(em.state_dict(), OUT_DIR / f"pilot_{tag}.pt")
    return r

In [14]:
def hazirla_rate(fp32_path, tag, step0=None, oran=0.57):
    """FP32 modeli bit aşamasına hazırlar. Adım = oran × notların std'si (ölçekten bağımsız birim).
    Pilotta 0.25 ≈ 0.57·std idi; 200 pair'de std 0.63 → adım ≈ 0.34."""
    sd = torch.load(fp32_path, map_location=device)
    if step0 is None:
        std = torch.cat([sd[f"grids.{k}"][PILOT].flatten() for k in GRID_LEVELS]).std().item()
        step0 = oran * std
    for k in GRID_LEVELS:
        sd[f"grids.{k}"] /= step0
        sd[f"inject.{k}.weight"] *= step0
    log_b = {str(k): torch.zeros(sd[f"grids.{k}"].shape[1], device=device) for k in GRID_LEVELS}
    yol = OUT_DIR / f"pilot_rate_{tag}_init.pt"
    torch.save({"model": sd, "log_b": log_b, "step0": step0}, yol)
    print(f"[{tag}] adım {step0:.3f} → {yol.name}")
    return yol

In [15]:
def run_rate(lam_max, epochs, init_path, tag, ramp_ep=50, eval_every=50, lr_carpan=1.0, grid_carpan=1.0,
             lam_min=3.0, batch=8, warm=50, soft=None, b_min=0.3, ctx=False, kuma=None, ema_ep=None):
    ck = torch.load(init_path, map_location=device)
    m = GridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device); m.load_state_dict(ck["model"])
    log_b = nn.ParameterDict({k: nn.Parameter(v.detach().clone().to(device)) for k, v in ck["log_b"].items()})
    ent = ContextEntropy().to(device) if ctx else None
    if ctx and ck.get("ent") is not None:
        ent.load_state_dict(ck["ent"])
    net_p = [p for n, p in m.named_parameters() if not n.startswith("grids")]
    grid_p = list(m.grids.parameters())
    gruplar = [{"params": net_p, "lr": 3e-4 * lr_carpan},
               {"params": grid_p, "lr": 3e-2 * lr_carpan * grid_carpan},
               {"params": list(log_b.parameters()), "lr": 1e-2 * lr_carpan}]
    if ctx:
        gruplar.append({"params": list(ent.parameters()), "lr": 3e-3 * lr_carpan})
    opt = torch.optim.Adam(gruplar)
    adim = math.ceil(len(PILOT) / batch)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / (warm * adim))
                                              * 0.5 * (1 + math.cos(math.pi * min(1, s / (epochs * adim)))))
    kirp = net_p + grid_p + (list(ent.parameters()) if ctx else [])
    ema = EMA(m, ema_ep * adim) if ema_ep else None                                     # v4
    print(f"[{tag}] bit aşaması | not LR {3e-2 * lr_carpan * grid_carpan:.2e} | ressam LR {3e-4 * lr_carpan:.2e}"
          f" | EMA {'yok' if not ema else f'{ema_ep} epoch'}")
    for ep in tqdm(range(epochs)):
        lam = lam_min + (lam_max - lam_min) * min(1, (ep + 1) / ramp_ep)
        t = ep / max(1, epochs - 1)
        alpha = 1 / (1 / soft[0] + (1 / soft[1] - 1 / soft[0]) * t) if soft else None   # C3: sıcaklık (1/alpha) doğrusal iner
        a_k = kuma[0] + (kuma[1] - kuma[0]) * t if kuma else None                          # Kumaraswamy a: 2 → 1
        perm = PILOT[torch.randperm(len(PILOT), device=device)]
        for i in range(0, len(perm), batch):
            ids = perm[i:i + batch]
            if kuma:
                fwd, noisy = grids_for_c3(m, ids, alpha, a_k)
            elif soft:
                fwd, noisy = grids_for_soft(m, ids, alpha)
            else:
                fwd, noisy = grids_for(m, ids, train=True)
            task, _, _ = task_loss(m(ids, grids=fwd), ids)
            if ctx:
                bits = sum(context_bits(ent, fwd, noisy).values())                 # bağlam: yuvarlanmış, bit: gürültülü
            else:
                bits = sum(laplace_bits(noisy[k], log_b[k].exp().clamp_min(b_min).view(1, -1, 1, 1)).sum()
                           for k in noisy)
            loss = task + lam * bits / len(ids) * NUM_PAIRS * BIT2SCORE
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(kirp, 1.0); opt.step(); sched.step()
            if ema: ema.guncelle(m)
        if (ep + 1) % eval_every == 0 or ep + 1 == epochs:
            em = ema.kopya(m) if ema else m                                               # ölçüm EMA kopyasıyla
            r = score(lambda ids: em(ids, grids=grids_for(em, ids, train=False)[0]), PILOT)
            q = {k: torch.round(g[PILOT].detach()) for k, g in em.grids.items()}
            kb_iid = sum(entropy_kb(v) for v in q.values()) * NUM_PAIRS / len(PILOT)
            kb = (sum(context_kb(ent, {k: g.detach() for k, g in em.grids.items()}, PILOT).values()) + ent_kb(ent)
                  if ctx else kb_iid)
            sifir = "/".join(f"%{100 * (v == 0).float().mean():.0f}" for v in q.values())
            print(f"[{tag}] E{ep + 1:03d} λ={lam:.0f} | dist {r['dist']:.4f} | ızgara ~{kb:.0f} KB"
                  f"{f' (bağımsız {kb_iid:.0f})' if ctx else ''} | sıfır {sifir} | toplam ≈ {r['dist'] + rate_of_kb(kb + NET_KB):.3f}"
                  f"{' [EMA]' if ema else ''}")
    em = ema.kopya(m) if ema else m
    torch.save({"model": em.state_dict(), "log_b": log_b.state_dict(), "step0": ck.get("step0"),
                "ent": ent.state_dict() if ctx else None}, OUT_DIR / f"pilot_rate_{tag}.pt")
    return r, kb


def net_ft(init_path, tag, epochs=200, batch=8, lr=3e-4, warm=10, ema_ep=None, qat=False, eval_every=50):
    """Notlar yuvarlanıp dondurulur, yalnız ressam eğitilir.
    v4: EMA (isteğe bağlı) · ressam QAT (isteğe bağlı) · her ölçümde 8 bit ressamla da ölçer ve
    8 bit dist'e göre EN İYİ checkpoint'i tutar (son epoch değil)."""
    ck = torch.load(init_path, map_location=device)
    m = GridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device); m.load_state_dict(ck["model"])
    with torch.no_grad():
        for g in m.grids.values(): g.round_()
    m.grids.requires_grad_(False)
    if ck.get("ent") is not None:
        ent = ContextEntropy().to(device); ent.load_state_dict(ck["ent"])
        kb = sum(context_kb(ent, dict(m.grids.items()), PILOT).values()) + ent_kb(ent)
    else:
        kb = sum(entropy_kb(m.grids[k][PILOT]) for k in m.grids) * NUM_PAIRS / len(PILOT)
    params = [p for n, p in m.named_parameters() if not n.startswith("grids")]
    adim = math.ceil(len(PILOT) / batch)
    opt = torch.optim.Adam(params, lr=lr)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / (warm * adim))
                                              * 0.5 * (1 + math.cos(math.pi * min(1, s / (epochs * adim)))))
    ema = EMA(m, ema_ep * adim) if ema_ep else None
    net_kb = ressam_kb(m)
    print(f"[{tag}] net_ft | batch {batch} | LR {lr:.2e} | EMA {'yok' if not ema else f'{ema_ep} epoch'} "
          f"| QAT {'8 bit' if qat else 'yok'} | notlar {kb:.0f} KB | ressam {net_kb:.0f} KB")
    en_iyi = {"d8": float("inf")}
    for ep in tqdm(range(epochs)):
        perm = PILOT[torch.randperm(len(PILOT), device=device)]
        for i in range(0, len(perm), batch):
            ids = perm[i:i + batch]
            asil = qat_ac(m) if qat else None
            loss, _, _ = task_loss(m(ids), ids)
            opt.zero_grad(set_to_none=True); loss.backward()
            if qat: qat_kapat(m, asil)                         # backward'dan sonra asıl ağırlıklara dön
            torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step()
            if ema: ema.guncelle(m)
        if (ep + 1) % eval_every == 0 or ep + 1 == epochs:
            em = ema.kopya(m) if ema else copy.deepcopy(m)
            r = score(em, PILOT)
            sd = copy.deepcopy(em.state_dict())                 # FP32 hali (kayıt için)
            quant_net_(em, 8)
            r8 = score(em, PILOT)                               # gerçek arşivdeki gibi 8 bit ressam
            if r8["dist"] < en_iyi["d8"]:
                en_iyi = {"d8": r8["dist"], "r": r, "r8": r8, "ep": ep + 1, "sd": sd}
            del em
            print(f"[{tag}] E{ep + 1:03d} | dist {r['dist']:.4f} | 8 bit {r8['dist']:.4f} | ızgara {kb:.0f} KB "
                  f"| toplam {r8['dist'] + rate_of_kb(kb + net_kb):.3f} | en iyi {en_iyi['d8']:.4f} (E{en_iyi['ep']})")
    m.load_state_dict(en_iyi["sd"])
    r, r8 = en_iyi["r"], en_iyi["r8"]
    print(f"[{tag}] SON | en iyi E{en_iyi['ep']} | dist {r['dist']:.4f} | 8 bit {r8['dist']:.4f} "
          f"(seg {r8['seg']:.4f}, pose {r8['pose']:.4f}) | notlar {kb:.0f} KB + ressam {net_kb:.0f} KB "
          f"| TOPLAM {r8['dist'] + rate_of_kb(kb + net_kb):.3f}")
    torch.save({"model": en_iyi["sd"], "log_b": ck["log_b"], "step0": ck.get("step0"), "ent": ck.get("ent")},
               OUT_DIR / f"pilot_rate_{tag}.pt")
    return m, r, kb, r8

## 8. Tezgâh döngüsü
Sıra listesindeki koşuları arka arkaya çalıştırır. Her koşu: (gerekirse FP32 sıfırdan) → `hazirla_rate` → bit aşaması → `net_ft`.
- Biten koşu `tezgah_sonuc.json`'a yazılır; döngü yeniden başlarsa atlanır. Bit aşamasının dosyası varsa doğrudan `net_ft`'ye geçer.
- Her satır `b2/tezgah_log.txt`'ye de yazılır. Hata olursa koşu atlanır, sıradakine geçilir.
- Oran = 8 bit dist / aynı pair'lerde başlangıç FP32 dist'i.
- **Koşu ayarları:** `ema_ep` (EMA ufku, epoch) · `not_lr` (bit aşamasında not LR çarpanı) · `qat` (net_ft'de 8 bit ressam) · `levels` / `widths` (mimari) · `fp32` (başlangıç dosyası) + `fp32_ep` (dosya yoksa sıfırdan kaç epoch) · `b_bit` / `b_ft` (batch'i elle ezmek için, ör. OOM olursa).

In [16]:
import sys, json, gc, time, traceback
from contextlib import redirect_stdout

LOG = OUT_DIR / "tezgah_log.txt"
SONUC = OUT_DIR / "tezgah_sonuc.json"
GECE = OUT_DIR / "gece_sonuc.json"                       # 600 pair gece kuyruğu (ayrı notebook'tan)
FP32_KAYNAK = "pilot_p600_uzun.pt"
VARSAYILAN = dict(levels={64: 2, 32: 2, 16: 2}, widths=[16, 16, 16, 16, 12, 12, 8])
REFERANS = {"p200u (ft900)":     ("sert", 30, 200, 0.2483, 94.6, 0.0898),
            "p600c3 (ft900)":    ("C3",   30, 600, 0.3033, 78.3, 0.0835),
            "p600sifir (ft900)": ("C3*",  30, 600, 0.3342, 65.1, 0.0835)}

BELLEK_GB = torch.cuda.get_device_properties(0).total_memory / 2 ** 30
B_BIT = 20
B_FT = 40 if BELLEK_GB > 30 else 20                      # L4 (22 GB) batch 40'ta OOM veriyordu
print(f"GPU belleği {BELLEK_GB:.0f} GB → bit aşaması batch {B_BIT}, net_ft batch {B_FT}")


class Tee:                                               # yazılanı hem ekrana hem dosyaya gönder
    def __init__(self, *akislar): self.akislar = akislar
    def write(self, s):
        for a in self.akislar: a.write(s); a.flush()
    def flush(self):
        for a in self.akislar: a.flush()


def mimari_kur(levels, widths, pairs):
    """Koşunun mimarisini ve pair sayısını globallere yazar (hazirla_rate / run_rate / net_ft bunları okur)."""
    global PILOT, GRID_LEVELS, WIDTHS, NET_KB
    PILOT = torch.arange(pairs, device=device)
    GRID_LEVELS, WIDTHS = dict(levels), list(widths)
    NET_KB = ressam_kb(GridDecoder(1, GRID_LEVELS, WIDTHS))


@torch.no_grad()
def fp32_ref(fp32):
    """Başlangıç FP32 modelinin aynı pair'lerdeki dist'i (oranın paydası)."""
    m = GridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device)
    m.load_state_dict(torch.load(OUT_DIR / fp32, map_location=device))
    d = score(m, PILOT)["dist"]
    del m; torch.cuda.empty_cache()
    return d


def gece_kosusu(T, lam, soft, kuma, pairs=120, epochs=1500, ramp=500, ft_ep=600,
                ema_ep=None, not_lr=1.0, qat=False, fp32=FP32_KAYNAK, fp32_ep=None,
                levels=None, widths=None, b_bit=None, b_ft=None):
    mimari_kur(levels or VARSAYILAN["levels"], widths or VARSAYILAN["widths"], pairs)
    print(f"\n[{T}] PILOT {pairs} pair | ızgaralar {GRID_LEVELS} | ressam {WIDTHS} ({NET_KB:.0f}K parametre)")
    if not (OUT_DIR / fp32).exists():
        if fp32_ep is None:
            raise FileNotFoundError(f"{fp32} yok ve fp32_ep verilmemiş")
        fp32_sifirdan(Path(fp32).stem.removeprefix("pilot_"), epochs=fp32_ep, batch=b_bit or B_BIT)
    ref = fp32_ref(fp32)
    print(f"[{T}] başlangıç {fp32}: FP32 dist {ref:.4f}")

    B = b_bit or B_BIT; s = (B / 8) ** 0.5
    if (OUT_DIR / f"pilot_rate_{T}.pt").exists():
        print(f"[{T}] bit aşaması zaten var, atlandı")
    else:
        init = hazirla_rate(OUT_DIR / fp32, T)
        run_rate(lam, epochs, init, T, ramp_ep=ramp, eval_every=250, batch=B, lr_carpan=s,
                 grid_carpan=0.55 / s * not_lr, b_min=0.3, soft=soft, kuma=kuma, ema_ep=ema_ep)

    B = b_ft or B_FT; s = (B / 8) ** 0.5
    _, r, kb, r8 = net_ft(OUT_DIR / f"pilot_rate_{T}.pt", f"{T}_ft{ft_ep}", epochs=ft_ep, batch=B,
                          lr=3e-4 * s, ema_ep=ema_ep, qat=qat)
    ayar = (("C3" if soft else "sert") + ("+ema" if ema_ep else "") + (f"+lr×{not_lr:.2g}" if not_lr != 1 else "")
            + ("+qat" if qat else "") + (f"+{len(GRID_LEVELS)}sv/{NET_KB:.0f}K" if levels or widths else ""))
    return {"ayar": ayar, "yontem": "C3" if soft else "sert", "lam": lam, "pairs": pairs,
            "dist": r["dist"], "dist8": r8["dist"], "seg8": r8["seg"], "pose8": r8["pose"],
            "kb": kb, "net_kb": NET_KB, "fp32_dist": ref, "toplam": r8["dist"] + rate_of_kb(kb + NET_KB)}


def ozet(sonuclar):
    satirlar = []
    for T, x in sonuclar.items():
        if T.startswith("_"): continue
        f = x.get("fp32_dist") or sonuclar.get(f"_fp32_{x['pairs']}", float("nan"))
        satirlar.append((T, x.get("ayar", x["yontem"]), x["lam"], x["pairs"], x["dist"], x.get("dist8"),
                         x["kb"], x.get("net_kb", 45), f))
    if GECE.exists():                                    # 600 pair gece kuyruğu (8 bit ölçümü yok)
        for T, x in json.loads(GECE.read_text()).items():
            satirlar.append((T, x["yontem"], x["lam"], 600, x["dist"], None, x["kb"], 45, 0.0835))
    satirlar += [(T, y, l, p, d, None, k, 45, f) for T, (y, l, p, d, k, f) in REFERANS.items()]
    toplam = lambda z: (z[5] if z[5] is not None else z[4]) + rate_of_kb(z[6] + z[7])
    print("\n=== ÖZET (toplama göre) ===")
    print(f"{'koşu':22s} {'ayar':20s} {'λ':>3s} {'pair':>4s} {'dist':>7s} {'8 bit':>7s} {'oran':>5s} "
          f"{'notKB':>6s} {'ağKB':>5s} {'toplam':>7s}")
    for z in sorted(satirlar, key=toplam):
        T, y, l, p, d, d8, k, nk, f = z
        d8s = f"{d8:7.4f}" if d8 is not None else f"{'–':>7s}"
        print(f"{T:22s} {y:20s} {l:3d} {p:4d} {d:7.4f} {d8s} {(d8 or d) / f:5.1f} {k:6.1f} {nk:5.0f} {toplam(z):7.3f}")
    print("toplam: 8 bit ölçümü varsa onunla, yoksa FP32 ressamlı dist ile (eski satırlar).")


def gece(sira):
    sonuclar = json.loads(SONUC.read_text()) if SONUC.exists() else {}
    with open(LOG, "a") as f, redirect_stdout(Tee(sys.stdout, f)):
        print(f"\n=== başladı {time.strftime('%Y-%m-%d %H:%M')} ===")
        for k in sira:
            if k["T"] in sonuclar:
                print(f"[{k['T']}] zaten bitmiş, atlandı"); continue
            t0 = time.time()
            try:
                sonuclar[k["T"]] = gece_kosusu(**k) | {"dk": round((time.time() - t0) / 60)}
                SONUC.write_text(json.dumps(sonuclar, indent=1))
                print(f"[{k['T']}] BİTTİ: {sonuclar[k['T']]}")
            except Exception:
                print(f"[{k['T']}] HATA:\n{traceback.format_exc()}")
            gc.collect(); torch.cuda.empty_cache()
        ozet(sonuclar)


C3 = dict(soft=(3.33, 10.0), kuma=(2.0, 1.0))
SERT = dict(soft=None, kuma=None)

GPU belleği 39 GB → bit aşaması batch 20, net_ft batch 40


### 8.1 Yarım kalanları tamamla (yalnız net_ft, koşu başına ~10–20 dk)
`t120u` ve `t120c3`'ün bit aşaması dosyaları duruyor; döngü onları atlayıp yalnız `net_ft`'yi koşar (en iyi checkpoint + 8 bit ölçüm ile).
**Kalibrasyon:** `t120u` ile gece kuyruğundaki `p600u` farkı ≲ 0.02 → 120 tezgâh geçerli.

In [ ]:
"""PILOT = torch.arange(120, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
WIDTHS = [16, 16, 16, 16, 12, 12, 8]               # 45K parametre

gece([dict(T="t120u",  lam=30, pairs=120, **SERT),
      dict(T="t120c3", lam=30, pairs=120, **C3)])"""

In [ ]:
"""PILOT = torch.arange(120, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
WIDTHS = [16, 16, 16, 16, 12, 12, 8]               # 45K parametre
mimari_kur(GRID_LEVELS, WIDTHS, 120)

# t120u'nun en iyi noktasından (0.281) devam: notlar tekrar serbest, sert + STE, λ30 baştan, LR ×0.3
B = B_BIT; s = (B / 8) ** 0.5
run_rate(30, 3000, OUT_DIR / "pilot_rate_t120u_ft600.pt", "t120u_devam",
         ramp_ep=1, lam_min=30, eval_every=250, batch=B,
         lr_carpan=0.3 * s, grid_carpan=0.55 / s, b_min=0.3)

B = B_FT; s = (B / 8) ** 0.5
_, r, kb, r8 = net_ft(OUT_DIR / "pilot_rate_t120u_devam.pt", "t120u_devam_ft600",
                      epochs=600, batch=B, lr=3e-4 * s)"""

In [ ]:
"""PILOT = torch.arange(120, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
WIDTHS = [16, 16, 16, 16, 12, 12, 8]               # 45K parametre
mimari_kur(GRID_LEVELS, WIDTHS, 120)

B = B_BIT; s = (B / 8) ** 0.5
run_rate(30, 3000, OUT_DIR / "pilot_rate_t120u_devam_ft600.pt", "t120u_devam2",
         ramp_ep=1, lam_min=30, eval_every=250, batch=B,
         lr_carpan=0.3 * s, grid_carpan=0.55 / s, b_min=0.3)

B = B_FT; s = (B / 8) ** 0.5
_, r, kb, r8 = net_ft(OUT_DIR / "pilot_rate_t120u_devam2.pt", "t120u_devam2_ft600",
                      epochs=1800, batch=B, lr=3e-4 * s)"""

'PILOT = torch.arange(120, device=device)\nGRID_LEVELS = {64: 2, 32: 2, 16: 2}\nWIDTHS = [16, 16, 16, 16, 12, 12, 8]               # 45K parametre\nmimari_kur(GRID_LEVELS, WIDTHS, 120)\n\nB = B_BIT; s = (B / 8) ** 0.5\nrun_rate(30, 3000, OUT_DIR / "pilot_rate_t120u_devam_ft600.pt", "t120u_devam2",\n         ramp_ep=1, lam_min=30, eval_every=250, batch=B,\n         lr_carpan=0.3 * s, grid_carpan=0.55 / s, b_min=0.3)\n\nB = B_FT; s = (B / 8) ** 0.5\n_, r, kb, r8 = net_ft(OUT_DIR / "pilot_rate_t120u_devam2.pt", "t120u_devam2_ft600",\n                      epochs=1800, batch=B, lr=3e-4 * s)'

In [17]:
PILOT = torch.arange(120, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
WIDTHS = [16, 16, 16, 16, 12, 12, 8]               # 45K parametre
mimari_kur(GRID_LEVELS, WIDTHS, 120)


@torch.no_grad()
def olc(yol):
    """Kayıtlı bir ft checkpoint'ini gerçek arşivdeki gibi ölçer: 8 bit ressam + yuvarlanmış notlar."""
    ck = torch.load(yol, map_location=device)
    m = GridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device)
    m.load_state_dict(ck["model"])
    kb = sum(entropy_kb(torch.round(m.grids[k][PILOT])) for k in m.grids) * NUM_PAIRS / len(PILOT)
    quant_net_(m, 8)
    d8 = score(m, PILOT)["dist"]
    del m; torch.cuda.empty_cache()
    return d8, kb, d8 + rate_of_kb(kb + NET_KB)


def devam_turlari(ilk=4, son=8, esik=0.005, bit_ep=3000, ft_ep=600):
    """devam{ilk}, devam{ilk+1}, … koşar. Her tur bir öncekinin ft checkpoint'inden başlar.
    Tur kazancı `esik`in altına düşünce durur. Biten turlar atlanır (hücreyi tekrar çalıştırmak kaldığı yerden sürdürür)."""
    sonuclar = json.loads(SONUC.read_text()) if SONUC.exists() else {}
    onceki = OUT_DIR / f"pilot_rate_t120u_devam{ilk - 1}_ft{ft_ep}.pt"
    with open(LOG, "a") as f, redirect_stdout(Tee(sys.stdout, f)):
        print(f"\n=== devam turları başladı {time.strftime('%Y-%m-%d %H:%M')} ===")
        _, _, t_onceki = olc(onceki)
        print(f"[başlangıç] {onceki.name}: toplam {t_onceki:.3f}")
        for n in range(ilk, son + 1):
            T = f"t120u_devam{n}"
            bit_yol = OUT_DIR / f"pilot_rate_{T}.pt"
            ft_yol = OUT_DIR / f"pilot_rate_{T}_ft{ft_ep}.pt"
            t0 = time.time()
            if ft_yol.exists():
                print(f"[{T}] zaten bitmiş, ölçülüp geçiliyor")
            else:
                if not bit_yol.exists():                         # bit aşaması: notlar serbest, LR ×0.3, λ30 sabit
                    B = B_BIT; s = (B / 8) ** 0.5
                    run_rate(30, bit_ep, onceki, T, ramp_ep=1, lam_min=30, eval_every=250, batch=B,
                             lr_carpan=0.3 * s, grid_carpan=0.55 / s, b_min=0.3)
                B = B_FT; s = (B / 8) ** 0.5                     # net_ft: notlar donuk, sadece ressam
                net_ft(bit_yol, f"{T}_ft{ft_ep}", epochs=ft_ep, batch=B, lr=3e-4 * s)
            d8, kb, t = olc(ft_yol)
            kazanc = t_onceki - t
            sonuclar[T] = {"ayar": f"devam{n}", "yontem": "sert", "lam": 30, "pairs": 120,
                           "dist": d8, "dist8": d8, "kb": kb, "net_kb": NET_KB,
                           "fp32_dist": sonuclar.get("_fp32_120", 0.0733), "toplam": t,
                           "dk": round((time.time() - t0) / 60)}
            SONUC.write_text(json.dumps(sonuclar, indent=1))
            print(f"[{T}] TUR SONU | 8 bit {d8:.4f} | notlar {kb:.0f} KB | toplam {t:.3f} | kazanç {kazanc:+.3f}")
            if kazanc < esik:
                print(f"[{T}] kazanç {esik} altında → turlar durdu. En iyi: {onceki.name if kazanc < 0 else ft_yol.name}")
                break
            onceki, t_onceki = ft_yol, t
            gc.collect(); torch.cuda.empty_cache()
        ozet(sonuclar)




# **600 pair**

In [18]:
PILOT = torch.arange(600, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
WIDTHS = [16, 16, 16, 16, 12, 12, 8]               # 45K parametre
mimari_kur(GRID_LEVELS, WIDTHS, 600)


def hiz_olc(B, adim=10):
    """Bit aşamasının bir adımını B batch ile ölçer → epoch süresi ve tepe bellek."""
    m = GridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device)
    m.load_state_dict(torch.load(OUT_DIR / FP32_KAYNAK, map_location=device))
    opt = torch.optim.Adam(m.parameters(), lr=1e-6)
    ids = PILOT[:B]
    torch.cuda.reset_peak_memory_stats()
    for i in range(adim + 2):                       # ilk 2 adım ısınma, ölçülmez
        if i == 2:
            torch.cuda.synchronize(); t0 = time.time()
        fwd, _ = grids_for(m, ids, train=True)
        loss, _, _ = task_loss(m(ids, grids=fwd), ids)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    torch.cuda.synchronize()
    s_adim = (time.time() - t0) / adim
    ep = s_adim * math.ceil(len(PILOT) / B)
    print(f"B={B:3d} | adım {s_adim * 1000:5.0f} ms | epoch {ep:5.2f} sn | 3000 epoch {ep * 3000 / 3600:4.1f} sa "
          f"| tepe bellek {torch.cuda.max_memory_allocated() / 2 ** 30:4.1f} GB")
    del m, opt; gc.collect(); torch.cuda.empty_cache()


for B in [20, 40, 60, 80, 100]:
    try:
        hiz_olc(B)
    except torch.OutOfMemoryError:
        print(f"B={B} OOM"); gc.collect(); torch.cuda.empty_cache(); break

B= 20 | adım   157 ms | epoch  4.71 sn | 3000 epoch  3.9 sa | tepe bellek 15.6 GB
B= 40 | adım   297 ms | epoch  4.45 sn | 3000 epoch  3.7 sa | tepe bellek 30.8 GB
B=60 OOM


In [19]:
def run_rate(lam_max, epochs, init_path, tag, ramp_ep=50, eval_every=50, lr_carpan=1.0, grid_carpan=1.0,
             lam_min=3.0, batch=8, warm=50, soft=None, b_min=0.3, ctx=False, kuma=None, ema_ep=None, birikim=1):
    """birikim > 1: `birikim` parça (her biri `batch` pair) biriktirilip tek Adam adımı atılır.
    Etkin batch = batch × birikim; bellek yalnız `batch` kadar."""
    ck = torch.load(init_path, map_location=device)
    m = GridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device); m.load_state_dict(ck["model"])
    log_b = nn.ParameterDict({k: nn.Parameter(v.detach().clone().to(device)) for k, v in ck["log_b"].items()})
    ent = ContextEntropy().to(device) if ctx else None
    if ctx and ck.get("ent") is not None:
        ent.load_state_dict(ck["ent"])
    net_p = [p for n, p in m.named_parameters() if not n.startswith("grids")]
    grid_p = list(m.grids.parameters())
    gruplar = [{"params": net_p, "lr": 3e-4 * lr_carpan},
               {"params": grid_p, "lr": 3e-2 * lr_carpan * grid_carpan},
               {"params": list(log_b.parameters()), "lr": 1e-2 * lr_carpan}]
    if ctx:
        gruplar.append({"params": list(ent.parameters()), "lr": 3e-3 * lr_carpan})
    opt = torch.optim.Adam(gruplar)
    adim = math.ceil(len(PILOT) / (batch * birikim))                   # epoch başına Adam adımı
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / (warm * adim))
                                              * 0.5 * (1 + math.cos(math.pi * min(1, s / (epochs * adim)))))
    kirp = net_p + grid_p + (list(ent.parameters()) if ctx else [])
    ema = EMA(m, ema_ep * adim) if ema_ep else None
    print(f"[{tag}] bit aşaması | parça {batch} × birikim {birikim} = etkin batch {batch * birikim} "
          f"| {adim} adım/epoch | not LR {3e-2 * lr_carpan * grid_carpan:.2e} | ressam LR {3e-4 * lr_carpan:.2e}")
    for ep in tqdm(range(epochs)):
        lam = lam_min + (lam_max - lam_min) * min(1, (ep + 1) / ramp_ep)
        t = ep / max(1, epochs - 1)
        alpha = 1 / (1 / soft[0] + (1 / soft[1] - 1 / soft[0]) * t) if soft else None
        a_k = kuma[0] + (kuma[1] - kuma[0]) * t if kuma else None
        perm = PILOT[torch.randperm(len(PILOT), device=device)]
        parcalar = [perm[i:i + batch] for i in range(0, len(perm), batch)]
        for j in range(0, len(parcalar), birikim):
            grup = parcalar[j:j + birikim]
            n_grup = sum(len(p) for p in grup)
            opt.zero_grad(set_to_none=True)
            for ids in grup:                                           # gradyanlar birikir, adım yok
                if kuma:
                    fwd, noisy = grids_for_c3(m, ids, alpha, a_k)
                elif soft:
                    fwd, noisy = grids_for_soft(m, ids, alpha)
                else:
                    fwd, noisy = grids_for(m, ids, train=True)
                task, _, _ = task_loss(m(ids, grids=fwd), ids)
                if ctx:
                    bits = sum(context_bits(ent, fwd, noisy).values())
                else:
                    bits = sum(laplace_bits(noisy[k], log_b[k].exp().clamp_min(b_min).view(1, -1, 1, 1)).sum()
                               for k in noisy)
                # büyük batch'in kaybıyla birebir aynı: görev kaybı pair ortalaması, bitler grup toplamı / grup boyu
                loss = (task * len(ids) + lam * bits * NUM_PAIRS * BIT2SCORE) / n_grup
                loss.backward()
            torch.nn.utils.clip_grad_norm_(kirp, 1.0); opt.step(); sched.step()
            if ema: ema.guncelle(m)
        if (ep + 1) % eval_every == 0 or ep + 1 == epochs:
            em = ema.kopya(m) if ema else m
            r = score(lambda ids: em(ids, grids=grids_for(em, ids, train=False)[0]), PILOT)
            q = {k: torch.round(g[PILOT].detach()) for k, g in em.grids.items()}
            kb_iid = sum(entropy_kb(v) for v in q.values()) * NUM_PAIRS / len(PILOT)
            kb = (sum(context_kb(ent, {k: g.detach() for k, g in em.grids.items()}, PILOT).values()) + ent_kb(ent)
                  if ctx else kb_iid)
            sifir = "/".join(f"%{100 * (v == 0).float().mean():.0f}" for v in q.values())
            print(f"[{tag}] E{ep + 1:03d} λ={lam:.0f} | dist {r['dist']:.4f} | ızgara ~{kb:.0f} KB"
                  f" | sıfır {sifir} | toplam ≈ {r['dist'] + rate_of_kb(kb + NET_KB):.3f}{' [EMA]' if ema else ''}")
    em = ema.kopya(m) if ema else m
    torch.save({"model": em.state_dict(), "log_b": log_b.state_dict(), "step0": ck.get("step0"),
                "ent": ent.state_dict() if ctx else None}, OUT_DIR / f"pilot_rate_{tag}.pt")
    return r, kb


def devam_turlari(on_ek, ilk=1, son=8, esik=0.005, bit_ep=3000, ft_ep=600, ft_eval=100,
                  parca=None, birikim=1):
    """Devam turları. Bit aşaması: `parca` × `birikim` etkin batch (LR etkin batch'e göre ölçeklenir)."""
    parca = parca or B_BIT
    sonuclar = json.loads(SONUC.read_text()) if SONUC.exists() else {}
    onceki = OUT_DIR / (f"pilot_rate_{on_ek}_ft{ft_ep}.pt" if ilk == 1
                        else f"pilot_rate_{on_ek}_devam{ilk - 1}_ft{ft_ep}.pt")
    with open(LOG, "a") as f, redirect_stdout(Tee(sys.stdout, f)):
        print(f"\n=== {on_ek} devam turları başladı {time.strftime('%Y-%m-%d %H:%M')} | {len(PILOT)} pair ===")
        _, _, t_onceki = olc(onceki)
        print(f"[başlangıç] {onceki.name}: toplam {t_onceki:.3f}")
        for n in range(ilk, son + 1):
            T = f"{on_ek}_devam{n}"
            bit_yol = OUT_DIR / f"pilot_rate_{T}.pt"
            ft_yol = OUT_DIR / f"pilot_rate_{T}_ft{ft_ep}.pt"
            t0 = time.time()
            if ft_yol.exists():
                print(f"[{T}] zaten bitmiş, ölçülüp geçiliyor")
            else:
                if not bit_yol.exists():
                    s = (parca * birikim / 8) ** 0.5                   # LR etkin batch'e göre
                    run_rate(30, bit_ep, onceki, T, ramp_ep=1, lam_min=30, eval_every=250, batch=parca,
                             birikim=birikim, lr_carpan=0.3 * s, grid_carpan=0.55 / s, b_min=0.3)
                B = B_FT; s = (B / 8) ** 0.5
                net_ft(bit_yol, f"{T}_ft{ft_ep}", epochs=ft_ep, batch=B, lr=3e-4 * s, eval_every=ft_eval)
            d8, kb, t = olc(ft_yol)
            kazanc = t_onceki - t
            sonuclar[T] = {"ayar": f"devam{n}", "yontem": "sert", "lam": 30, "pairs": len(PILOT),
                           "dist": d8, "dist8": d8, "kb": kb, "net_kb": NET_KB,
                           "fp32_dist": sonuclar.get(f"_fp32_{len(PILOT)}", 0.0835), "toplam": t,
                           "dk": round((time.time() - t0) / 60)}
            SONUC.write_text(json.dumps(sonuclar, indent=1))
            print(f"[{T}] TUR SONU | 8 bit {d8:.4f} | notlar {kb:.0f} KB | toplam {t:.3f} | kazanç {kazanc:+.3f}")
            if kazanc < esik:
                print(f"[{T}] kazanç {esik} altında → turlar durdu. En iyi: {onceki.name if kazanc < 0 else ft_yol.name}")
                break
            onceki, t_onceki = ft_yol, t
            gc.collect(); torch.cuda.empty_cache()
        ozet(sonuclar)

In [20]:
PILOT = torch.arange(600, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
WIDTHS = [16, 16, 16, 16, 12, 12, 8]               # 45K parametre
mimari_kur(GRID_LEVELS, WIDTHS, 600)


@torch.no_grad()
def olc(yol):
    """Kayıtlı bir ft checkpoint'ini gerçek arşivdeki gibi ölçer: 8 bit ressam + yuvarlanmış notlar."""
    ck = torch.load(yol, map_location=device)
    m = GridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device)
    m.load_state_dict(ck["model"])
    kb = sum(entropy_kb(torch.round(m.grids[k][PILOT])) for k in m.grids) * NUM_PAIRS / len(PILOT)
    quant_net_(m, 8)
    d8 = score(m, PILOT)["dist"]
    del m; torch.cuda.empty_cache()
    return d8, kb, d8 + rate_of_kb(kb + NET_KB)


def devam_turlari(on_ek, ilk=1, son=8, esik=0.005, bit_ep=3000, ft_ep=600, ft_eval=100):
    """{on_ek}_devam{ilk}, … koşar; her tur bir öncekinin ft checkpoint'inden başlar.
    ilk=1 ise başlangıç {on_ek}_ft600'dür. Kazanç `esik`in altına düşünce durur.
    Biten turlar atlanır: hücreyi yeniden çalıştırmak kaldığı yerden sürdürür."""
    sonuclar = json.loads(SONUC.read_text()) if SONUC.exists() else {}
    onceki = OUT_DIR / (f"pilot_rate_{on_ek}_ft{ft_ep}.pt" if ilk == 1
                        else f"pilot_rate_{on_ek}_devam{ilk - 1}_ft{ft_ep}.pt")
    with open(LOG, "a") as f, redirect_stdout(Tee(sys.stdout, f)):
        print(f"\n=== {on_ek} devam turları başladı {time.strftime('%Y-%m-%d %H:%M')} | {len(PILOT)} pair ===")
        _, _, t_onceki = olc(onceki)
        print(f"[başlangıç] {onceki.name}: toplam {t_onceki:.3f}")
        for n in range(ilk, son + 1):
            T = f"{on_ek}_devam{n}"
            bit_yol = OUT_DIR / f"pilot_rate_{T}.pt"
            ft_yol = OUT_DIR / f"pilot_rate_{T}_ft{ft_ep}.pt"
            t0 = time.time()
            if ft_yol.exists():
                print(f"[{T}] zaten bitmiş, ölçülüp geçiliyor")
            else:
                if not bit_yol.exists():                         # bit aşaması: notlar serbest, LR ×0.3, λ30 sabit
                    B = B_BIT; s = (B / 8) ** 0.5
                    run_rate(30, bit_ep, onceki, T, ramp_ep=1, lam_min=30, eval_every=250, batch=B,
                             lr_carpan=0.3 * s, grid_carpan=0.55 / s, b_min=0.3)
                B = B_FT; s = (B / 8) ** 0.5                     # net_ft: notlar donuk, sadece ressam
                net_ft(bit_yol, f"{T}_ft{ft_ep}", epochs=ft_ep, batch=B, lr=3e-4 * s, eval_every=ft_eval)
            d8, kb, t = olc(ft_yol)
            kazanc = t_onceki - t
            sonuclar[T] = {"ayar": f"devam{n}", "yontem": "sert", "lam": 30, "pairs": len(PILOT),
                           "dist": d8, "dist8": d8, "kb": kb, "net_kb": NET_KB,
                           "fp32_dist": sonuclar.get(f"_fp32_{len(PILOT)}", 0.0835), "toplam": t,
                           "dk": round((time.time() - t0) / 60)}
            SONUC.write_text(json.dumps(sonuclar, indent=1))
            print(f"[{T}] TUR SONU | 8 bit {d8:.4f} | notlar {kb:.0f} KB | toplam {t:.3f} | kazanç {kazanc:+.3f}")
            if kazanc < esik:
                print(f"[{T}] kazanç {esik} altında → turlar durdu. En iyi: {onceki.name if kazanc < 0 else ft_yol.name}")
                break
            onceki, t_onceki = ft_yol, t
            gc.collect(); torch.cuda.empty_cache()
        ozet(sonuclar)

"""
# 1) Başlangıç turu: t120u'nun aynısı, 600 pair (pilot_p600_uzun.pt'den; bit 1500 rampa 500 + ft 600)
gece([dict(T="p600u", lam=30, pairs=600, **SERT)])

# 2) Devam turları: p600u_devam1, devam2, …
devam_turlari(on_ek="p600u", ilk=1, son=8)"""

'\n# 1) Başlangıç turu: t120u\'nun aynısı, 600 pair (pilot_p600_uzun.pt\'den; bit 1500 rampa 500 + ft 600)\ngece([dict(T="p600u", lam=30, pairs=600, **SERT)])\n\n# 2) Devam turları: p600u_devam1, devam2, …\ndevam_turlari(on_ek="p600u", ilk=1, son=8)'

In [21]:
PILOT = torch.arange(600, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
WIDTHS = [16, 16, 16, 16, 12, 12, 8]               # eski ressam: 45K parametre
mimari_kur(GRID_LEVELS, WIDTHS, 600)

YENI_WIDTHS = [20, 20, 20, 20, 16, 16, 10]         # yeni ressam: ~73K parametre (+27.5 KB ≈ +0.018 rate)


@torch.no_grad()
def genislet(eski_yol, yeni_widths, tag):
    """Ressamı işlevi koruyarak genişletir (Net2Net tarzı).
    Eski kanallar aynen kopyalanır. Yeni kanalların GİRİŞ ağırlıkları rastgele kalır (öğrenebilsinler),
    eski çıkışlara giden ağırlıkları 0 yapılır → başlangıçta çıktı birebir aynı."""
    ck = torch.load(eski_yol, map_location=device)
    eski = GridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device)
    eski.load_state_dict(ck["model"])
    torch.manual_seed(0)
    yeni = GridDecoder(NUM_PAIRS, GRID_LEVELS, yeni_widths).to(device)
    for k in eski.grids:
        yeni.grids[k].copy_(eski.grids[k])

    def kopyala(e, y, n_out, n_in):
        y.weight[:n_out, n_in:] = 0                  # yeni kanallardan eski çıkışlara: 0
        y.weight[:n_out, :n_in] = e.weight           # eski → eski: aynen
        y.bias[:n_out] = e.bias

    for d in eski.inject:                            # notlar → ressam (giriş kanalı sayısı değişmiyor)
        n = eski.inject[d].weight.shape[0]
        yeni.inject[d].weight[:n] = eski.inject[d].weight
        yeni.inject[d].bias[:n] = eski.inject[d].bias
    for i in range(6):
        kopyala(eski.ups[i], yeni.ups[i], 4 * WIDTHS[i + 1], WIDTHS[i])   # pixel_shuffle: kanal c → satır 4c..4c+3
        kopyala(eski.skips[i], yeni.skips[i], WIDTHS[i + 1], WIDTHS[i])
    kopyala(eski.head, yeni.head, 6, WIDTHS[6])

    ids = PILOT[:8]                                  # sağlama: çıktı aynı mı?
    fark = (eski(ids) - yeni(ids)).abs().max().item()
    n_eski = sum(p.numel() for n, p in eski.named_parameters() if not n.startswith("grids"))
    n_yeni = sum(p.numel() for n, p in yeni.named_parameters() if not n.startswith("grids"))
    print(f"[{tag}] genişletme: {n_eski:,} → {n_yeni:,} parametre | çıktı farkı {fark:.2e} (≈0 olmalı)")
    torch.save({"model": yeni.state_dict(), "log_b": ck["log_b"], "step0": ck.get("step0"), "ent": ck.get("ent")},
               OUT_DIR / f"pilot_rate_{tag}.pt")
    del eski, yeni; torch.cuda.empty_cache()


B = B_FT; s = (B / 8) ** 0.5

"""# 1) KONTROL: genişletme yok, aynı ft (sadece ekstra ft'nin etkisi)
mimari_kur(GRID_LEVELS, WIDTHS, 600)
net_ft(OUT_DIR / "pilot_rate_p600u_devam3_ft600.pt", "p600u_d3_kontrol_ft600",
       epochs=600, batch=B, lr=3e-4 * s, eval_every=100)"""

# 2) GENİŞ: aynı başlangıç, ressam ~73K
genislet(OUT_DIR / "pilot_rate_p600u_devam3_ft600.pt", YENI_WIDTHS, "p600u_d3_genis")
mimari_kur(GRID_LEVELS, YENI_WIDTHS, 600)          # bundan sonra her şey yeni ressamla (NET_KB de güncellenir)
net_ft(OUT_DIR / "pilot_rate_p600u_d3_genis.pt", "p600u_d3_genis_ft600",
       epochs=600, batch=B, lr=3e-4 * s, eval_every=100)

[p600u_d3_genis] genişletme: 45,382 → 72,868 parametre | çıktı farkı 1.98e-01 (≈0 olmalı)
[p600u_d3_genis_ft600] net_ft | batch 40 | LR 6.71e-04 | EMA yok | QAT yok | notlar 52 KB | ressam 73 KB


  0%|          | 0/600 [00:00<?, ?it/s]

[p600u_d3_genis_ft600] E100 | dist 0.2806 | 8 bit 0.2795 | ızgara 52 KB | toplam 0.363 | en iyi 0.2795 (E100)
[p600u_d3_genis_ft600] E200 | dist 0.2819 | 8 bit 0.2870 | ızgara 52 KB | toplam 0.370 | en iyi 0.2795 (E100)
[p600u_d3_genis_ft600] E300 | dist 0.2537 | 8 bit 0.2590 | ızgara 52 KB | toplam 0.342 | en iyi 0.2590 (E300)
[p600u_d3_genis_ft600] E400 | dist 0.2448 | 8 bit 0.2528 | ızgara 52 KB | toplam 0.336 | en iyi 0.2528 (E400)
[p600u_d3_genis_ft600] E500 | dist 0.2367 | 8 bit 0.2435 | ızgara 52 KB | toplam 0.327 | en iyi 0.2435 (E500)
[p600u_d3_genis_ft600] E600 | dist 0.2354 | 8 bit 0.2433 | ızgara 52 KB | toplam 0.326 | en iyi 0.2433 (E600)
[p600u_d3_genis_ft600] SON | en iyi E600 | dist 0.2354 | 8 bit 0.2433 (seg 0.2113, pose 0.0320) | notlar 52 KB + ressam 73 KB | TOPLAM 0.326


(GridDecoder(
   (grids): ParameterDict(
       (16): Parameter containing: [torch.cuda.FloatTensor of size 600x2x24x32 (cuda:0)]
       (32): Parameter containing: [torch.cuda.FloatTensor of size 600x2x12x16 (cuda:0)]
       (64): Parameter containing: [torch.cuda.FloatTensor of size 600x2x6x8 (cuda:0)]
   )
   (inject): ModuleDict(
     (64): Conv2d(2, 20, kernel_size=(1, 1), stride=(1, 1))
     (32): Conv2d(2, 20, kernel_size=(1, 1), stride=(1, 1))
     (16): Conv2d(2, 20, kernel_size=(1, 1), stride=(1, 1))
   )
   (ups): ModuleList(
     (0-2): 3 x Conv2d(20, 80, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
     (3): Conv2d(20, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
     (4): Conv2d(16, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
     (5): Conv2d(16, 40, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   )
   (skips): ModuleList(
     (0-2): 3 x Conv2d(20, 20, kernel_size=(1, 1), stride=(1, 1))
     (3): Conv2d(20, 16, kernel_size=(1, 1), stride=

In [22]:
PILOT = torch.arange(600, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
WIDTHS = [16, 16, 16, 16, 12, 12, 8]               # kontrol eski ressamla eğitildi
mimari_kur(GRID_LEVELS, WIDTHS, 600)

for ad in ["p600u_devam3_ft600", "p600u_d3_kontrol_ft600"]:
    d8, kb, top = olc(OUT_DIR / f"pilot_rate_{ad}.pt")
    print(f"{ad:28s} | 8 bit dist {d8:.4f} | notlar {kb:.1f} KB | TOPLAM {top:.3f}")

p600u_devam3_ft600           | 8 bit dist 0.2636 | notlar 51.9 KB | TOPLAM 0.328
p600u_d3_kontrol_ft600       | 8 bit dist 0.2604 | notlar 51.9 KB | TOPLAM 0.325


In [24]:
def ressam_entropi(yol, bitler=(8, 6)):
    """quant_net_ ile aynı format: conv ağırlıkları kanal başına simetrik, bias + kanal ölçekleri fp16."""
    sd = torch.load(yol, map_location="cpu")["model"]
    n = sum(v.numel() for k, v in sd.items() if not k.startswith("grids"))
    sonuc = {}
    for b in bitler:
        qmax = 2 ** (b - 1) - 1
        bit = 0.0
        for k, v in sd.items():
            if k.startswith("grids"):
                continue
            v = v.float()
            if v.dim() == 4:                                    # conv ağırlığı
                s = v.abs().amax(dim=(1, 2, 3), keepdim=True).clamp_min(1e-12) / qmax
                kod = torch.round(v / s).clamp(-qmax, qmax).long().flatten()
                p = torch.bincount(kod + qmax).float(); p = p[p > 0] / kod.numel()
                bit += -(p * p.log2()).sum().item() * kod.numel()
                bit += 16 * v.shape[0]                          # kanal başına fp16 ölçek
            else:                                               # bias
                bit += 16 * v.numel()
        sonuc[b] = bit / 8 / 1000
    print(f"{yol.name:40s} | {n:,} param | ham {n/1000:.1f} KB | "
          + " | ".join(f"{b} bit {kb:.1f} KB" for b, kb in sonuc.items()))

for ad in ["p600u_d3_kontrol_ft600", "p600u_d3_genis_ft600"]:
    ressam_entropi(OUT_DIR / f"pilot_rate_{ad}.pt")

pilot_rate_p600u_d3_kontrol_ft600.pt     | 45,382 param | ham 45.4 KB | 8 bit 43.4 KB | 6 bit 32.3 KB
pilot_rate_p600u_d3_genis_ft600.pt       | 72,868 param | ham 72.9 KB | 8 bit 67.0 KB | 6 bit 49.2 KB


In [25]:
PILOT = torch.arange(600, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
ESKI_W, YENI_W = [16, 16, 16, 16, 12, 12, 8], [20, 20, 20, 20, 16, 16, 10]


def ressam_kb_bit(sd, b):
    """quant_net_ formatı: conv ağırlıkları kanal başına simetrik b bit (entropi), bias + ölçekler fp16."""
    qmax, bit = 2 ** (b - 1) - 1, 0.0
    for k, v in sd.items():
        if k.startswith("grids"):
            continue
        v = v.float().cpu()
        if v.dim() == 4:
            s = v.abs().amax(dim=(1, 2, 3), keepdim=True).clamp_min(1e-12) / qmax
            kod = torch.round(v / s).clamp(-qmax, qmax).long().flatten()
            p = torch.bincount(kod + qmax).float(); p = p[p > 0] / kod.numel()
            bit += -(p * p.log2()).sum().item() * kod.numel() + 16 * v.shape[0]
        else:
            bit += 16 * v.numel()
    return bit / 8 / 1000


def bit_tarama(ad, widths, bitler=(8, 7, 6, 5)):
    mimari_kur(GRID_LEVELS, widths, 600)
    ck = torch.load(OUT_DIR / f"pilot_rate_{ad}.pt", map_location=device)
    for b in bitler:
        m = GridDecoder(NUM_PAIRS, GRID_LEVELS, widths).to(device)
        m.load_state_dict(ck["model"])
        with torch.no_grad():
            for g in m.grids.values(): g.round_()
            quant_net_(m, b)
            not_kb = sum(entropy_kb(m.grids[k][PILOT]) for k in m.grids)
            r = score(m, PILOT)
        r_kb = ressam_kb_bit(ck["model"], b)
        print(f"{ad:24s} | {b} bit | dist {r['dist']:.4f} (seg {r['seg']:.4f}) | "
              f"notlar {not_kb:.1f} + ressam {r_kb:.1f} KB | TOPLAM {r['dist'] + rate_of_kb(not_kb + r_kb):.3f}")
        del m; torch.cuda.empty_cache()


bit_tarama("p600u_d3_kontrol_ft600", ESKI_W)
bit_tarama("p600u_d3_genis_ft600", YENI_W)

p600u_d3_kontrol_ft600   | 8 bit | dist 0.2604 (seg 0.2281) | notlar 51.9 + ressam 43.4 KB | TOPLAM 0.324
p600u_d3_kontrol_ft600   | 7 bit | dist 0.2743 (seg 0.2382) | notlar 51.9 + ressam 37.9 KB | TOPLAM 0.334
p600u_d3_kontrol_ft600   | 6 bit | dist 0.3494 (seg 0.2804) | notlar 51.9 + ressam 32.3 KB | TOPLAM 0.405
p600u_d3_kontrol_ft600   | 5 bit | dist 0.6067 (seg 0.4593) | notlar 51.9 + ressam 26.5 KB | TOPLAM 0.659
p600u_d3_genis_ft600     | 8 bit | dist 0.2433 (seg 0.2113) | notlar 51.9 + ressam 67.0 KB | TOPLAM 0.322
p600u_d3_genis_ft600     | 7 bit | dist 0.2660 (seg 0.2237) | notlar 51.9 + ressam 58.1 KB | TOPLAM 0.339
p600u_d3_genis_ft600     | 6 bit | dist 0.3339 (seg 0.2762) | notlar 51.9 + ressam 49.2 KB | TOPLAM 0.401
p600u_d3_genis_ft600     | 5 bit | dist 0.6753 (seg 0.3965) | notlar 51.9 + ressam 40.0 KB | TOPLAM 0.736


In [26]:
PILOT = torch.arange(600, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
YENI_W = [20, 20, 20, 20, 16, 16, 10]
mimari_kur(GRID_LEVELS, YENI_W, 600)


def net_ft_q(init_path, tag, bits=8, epochs=600, batch=B_FT, lr=3e-4 * (B_FT / 8) ** 0.5, warm=10, eval_every=100):
    """net_ft + QAT, `bits` bitlik ressamla. Eğitim yuvarlanmış ağırlıkla ileri/geri, adım FP32 ağırlığa (STE).
    En iyi checkpoint `bits` bitlik dist'e göre; ressam KB'ı gerçek formatla (kanal başına entropi + fp16 bias/ölçek)."""
    ck = torch.load(init_path, map_location=device)
    m = GridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device); m.load_state_dict(ck["model"])
    with torch.no_grad():
        for g in m.grids.values(): g.round_()
    m.grids.requires_grad_(False)
    kb = sum(entropy_kb(m.grids[k][PILOT]) for k in m.grids)
    params = [p for n, p in m.named_parameters() if not n.startswith("grids")]
    adim = math.ceil(len(PILOT) / batch)
    opt = torch.optim.Adam(params, lr=lr)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / (warm * adim))
                                              * 0.5 * (1 + math.cos(math.pi * min(1, s / (epochs * adim)))))
    print(f"[{tag}] net_ft_q | QAT {bits} bit | batch {batch} | LR {lr:.2e} | notlar {kb:.1f} KB "
          f"| ressam başta {ressam_kb_bit(ck['model'], bits):.1f} KB")
    en_iyi = {"dq": float("inf")}
    for ep in tqdm(range(epochs)):
        perm = PILOT[torch.randperm(len(PILOT), device=device)]
        for i in range(0, len(perm), batch):
            ids = perm[i:i + batch]
            asil = qat_ac(m, bits)
            loss, _, _ = task_loss(m(ids), ids)
            opt.zero_grad(set_to_none=True); loss.backward()
            qat_kapat(m, asil)
            torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step()
        if (ep + 1) % eval_every == 0 or ep + 1 == epochs:
            em = copy.deepcopy(m)
            sd = copy.deepcopy(em.state_dict())
            r_kb = ressam_kb_bit(sd, bits)
            with torch.no_grad():
                quant_net_(em, bits)
            rq = score(em, PILOT)
            top = rq["dist"] + rate_of_kb(kb + r_kb)
            if top < en_iyi.get("top", float("inf")):
                en_iyi = {"dq": rq["dist"], "top": top, "rq": rq, "ep": ep + 1, "sd": sd, "r_kb": r_kb}
            del em
            print(f"[{tag}] E{ep + 1:03d} | {bits} bit dist {rq['dist']:.4f} (seg {rq['seg']:.4f}) "
                  f"| ressam {r_kb:.1f} KB | toplam {top:.3f} | en iyi {en_iyi['top']:.3f} (E{en_iyi['ep']})")
    rq = en_iyi["rq"]
    print(f"[{tag}] SON | en iyi E{en_iyi['ep']} | {bits} bit dist {rq['dist']:.4f} (seg {rq['seg']:.4f}, "
          f"pose {rq['pose']:.4f}) | notlar {kb:.1f} + ressam {en_iyi['r_kb']:.1f} KB | TOPLAM {en_iyi['top']:.3f}")
    torch.save({"model": en_iyi["sd"], "log_b": ck["log_b"], "step0": ck.get("step0"), "ent": ck.get("ent"),
                "bits": bits}, OUT_DIR / f"pilot_rate_{tag}.pt")
    del m; torch.cuda.empty_cache()


net_ft_q(OUT_DIR / "pilot_rate_p600u_d3_genis_ft600.pt", "p600u_genis_qat8", bits=8)
net_ft_q(OUT_DIR / "pilot_rate_p600u_d3_genis_ft600.pt", "p600u_genis_qat6", bits=6)

[p600u_genis_qat8] net_ft_q | QAT 8 bit | batch 40 | LR 6.71e-04 | notlar 51.9 KB | ressam başta 67.0 KB


  0%|          | 0/600 [00:00<?, ?it/s]

[p600u_genis_qat8] E100 | 8 bit dist 0.3019 (seg 0.2267) | ressam 67.4 KB | toplam 0.381 | en iyi 0.381 (E100)
[p600u_genis_qat8] E200 | 8 bit dist 0.2977 (seg 0.2227) | ressam 67.4 KB | toplam 0.377 | en iyi 0.377 (E200)
[p600u_genis_qat8] E300 | 8 bit dist 0.2688 (seg 0.2154) | ressam 67.4 KB | toplam 0.348 | en iyi 0.348 (E300)
[p600u_genis_qat8] E400 | 8 bit dist 0.2616 (seg 0.2123) | ressam 67.4 KB | toplam 0.341 | en iyi 0.341 (E400)
[p600u_genis_qat8] E500 | 8 bit dist 0.2480 (seg 0.2081) | ressam 67.4 KB | toplam 0.327 | en iyi 0.327 (E500)
[p600u_genis_qat8] E600 | 8 bit dist 0.2339 (seg 0.2032) | ressam 67.4 KB | toplam 0.313 | en iyi 0.313 (E600)
[p600u_genis_qat8] SON | en iyi E600 | 8 bit dist 0.2339 (seg 0.2032, pose 0.0307) | notlar 51.9 + ressam 67.4 KB | TOPLAM 0.313
[p600u_genis_qat6] net_ft_q | QAT 6 bit | batch 40 | LR 6.71e-04 | notlar 51.9 KB | ressam başta 49.2 KB


  0%|          | 0/600 [00:00<?, ?it/s]

[p600u_genis_qat6] E100 | 6 bit dist 0.4026 (seg 0.2972) | ressam 49.4 KB | toplam 0.470 | en iyi 0.470 (E100)
[p600u_genis_qat6] E200 | 6 bit dist 0.3824 (seg 0.3000) | ressam 49.4 KB | toplam 0.450 | en iyi 0.450 (E200)
[p600u_genis_qat6] E300 | 6 bit dist 0.3846 (seg 0.2813) | ressam 49.4 KB | toplam 0.452 | en iyi 0.450 (E200)
[p600u_genis_qat6] E400 | 6 bit dist 0.3566 (seg 0.2731) | ressam 49.4 KB | toplam 0.424 | en iyi 0.424 (E400)
[p600u_genis_qat6] E500 | 6 bit dist 0.3183 (seg 0.2624) | ressam 49.4 KB | toplam 0.386 | en iyi 0.386 (E500)
[p600u_genis_qat6] E600 | 6 bit dist 0.2618 (seg 0.2243) | ressam 49.4 KB | toplam 0.329 | en iyi 0.329 (E600)
[p600u_genis_qat6] SON | en iyi E600 | 6 bit dist 0.2618 (seg 0.2243, pose 0.0375) | notlar 51.9 + ressam 49.4 KB | TOPLAM 0.329


In [27]:
mimari_kur(GRID_LEVELS, YENI_W, 600)
net_ft_q(OUT_DIR / "pilot_rate_p600u_genis_qat8.pt", "p600u_genis_qat8_lr15", bits=8, lr=1.5e-4)

[p600u_genis_qat8_lr15] net_ft_q | QAT 8 bit | batch 40 | LR 1.50e-04 | notlar 51.9 KB | ressam başta 67.4 KB


  0%|          | 0/600 [00:00<?, ?it/s]

[p600u_genis_qat8_lr15] E100 | 8 bit dist 0.2621 (seg 0.2128) | ressam 67.4 KB | toplam 0.342 | en iyi 0.342 (E100)
[p600u_genis_qat8_lr15] E200 | 8 bit dist 0.2630 (seg 0.2105) | ressam 67.4 KB | toplam 0.342 | en iyi 0.342 (E100)
[p600u_genis_qat8_lr15] E300 | 8 bit dist 0.2547 (seg 0.2111) | ressam 67.4 KB | toplam 0.334 | en iyi 0.334 (E300)
[p600u_genis_qat8_lr15] E400 | 8 bit dist 0.2455 (seg 0.2070) | ressam 67.4 KB | toplam 0.325 | en iyi 0.325 (E400)
[p600u_genis_qat8_lr15] E500 | 8 bit dist 0.2425 (seg 0.2067) | ressam 67.4 KB | toplam 0.322 | en iyi 0.322 (E500)
[p600u_genis_qat8_lr15] E600 | 8 bit dist 0.2325 (seg 0.2021) | ressam 67.4 KB | toplam 0.312 | en iyi 0.312 (E600)
[p600u_genis_qat8_lr15] SON | en iyi E600 | 8 bit dist 0.2325 (seg 0.2021, pose 0.0304) | notlar 51.9 + ressam 67.4 KB | TOPLAM 0.312
